In [ ]:
import pytesseract
import re
from PIL import Image
from pathlib import Path
from collections import defaultdict

print(f"Tesseract Version: {pytesseract.get_tesseract_version()}")

raw_dir = Path('../../data/01_raw')
raw_dir.mkdir(parents=True, exist_ok=True)
ocr_dir = Path('../../data/02_ocr')
ocr_dir.mkdir(parents=True, exist_ok=True)


# OEM => OCR Engine Mode (0-3) 3 erkennt automatisch was es brauch
# PSM => Page Segmentation Mode (1-13)
    # 1 = vollautomatische seitenaalyse mit osd
    # 3 = vollautomatische seitenaalyse ohne osd (standardwert)
    # 4 = einzelne TExtplatze mit variablen Schriftgrößen
    # 6 = einzelnen, textblock
    # 11 = findet so viel text wie möglich (sparse text
custom_config = r'--oem 3 --psm 6'

# collect Data and group
documents = defaultdict(list)

for img_path in raw_dir.glob('*.jpg'):
    # ^(.*?) nimmt alles von anfang an als Gruppe 1 (basisname)
    # (\d+)$ nimmt alle ziffern am ende des namens als gruppe 2 (seitenzahl)
    match = re.match(r"^(.*?)(\d+)$", img_path.stem)

    if match:
        base_name = match.group(1)
        documents[base_name].append(img_path)
    else:
        print(f"Filename {img_path.name} does not match the expected pattern.")


# iterate group documents
for base_name, files in documents.items():
    # important: file sort (logic order)
    files.sort(key=lambda p: int(re.match(r"^(.*?)(\d+)$", p.stem).group(2)))

    doc_name = base_name.rstrip('_')
    out_file = ocr_dir / f"{doc_name}.txt"
    
    print(f"Verarbeite Dokument: {doc_name} ({len(files)} Seiten) -> {out_file.name}")

    with open(out_file, mode="w", encoding="utf-8") as f:
        
        for page_path in files:
            
            with Image.open(page_path) as img:
                text = pytesseract.image_to_string(img, lang="deu_frak", config=custom_config)

                # separator
                f.write(f"\n{'='*20}\n--- {page_path.name} ---\n{'='*20}\n\n")
                f.write(text)
                f.write("\n")

print("OCR done")
    

# Pandas Dataframe mit Bounding Boxes und COnfidence Score
#test = pytesseract.image_to_data(img, lang='deu', config=custom_config)
#print (test)


Tesseract Version: 5.5.2
defaultdict(<class 'list'>, {'R_9346_I_1_': [PosixPath('../../data/01_raw/R_9346_I_1_0002.jpg'), PosixPath('../../data/01_raw/R_9346_I_1_0004.jpg'), PosixPath('../../data/01_raw/R_9346_I_1_0005.jpg'), PosixPath('../../data/01_raw/R_9346_I_1_0006.jpg'), PosixPath('../../data/01_raw/R_9346_I_1_0003.jpg'), PosixPath('../../data/01_raw/R_9346_I_1_0001.jpg')], 'R_9346_I_2_': [PosixPath('../../data/01_raw/R_9346_I_2_0009.jpg'), PosixPath('../../data/01_raw/R_9346_I_2_0005.jpg'), PosixPath('../../data/01_raw/R_9346_I_2_0001.jpg'), PosixPath('../../data/01_raw/R_9346_I_2_0010.jpg'), PosixPath('../../data/01_raw/R_9346_I_2_0003.jpg'), PosixPath('../../data/01_raw/R_9346_I_2_0008.jpg'), PosixPath('../../data/01_raw/R_9346_I_2_0006.jpg'), PosixPath('../../data/01_raw/R_9346_I_2_0007.jpg'), PosixPath('../../data/01_raw/R_9346_I_2_0002.jpg'), PosixPath('../../data/01_raw/R_9346_I_2_0004.jpg')]})
Verarbeite Dokument: R_9346_I_1 (6 Seiten) -> R_9346_I_1.txt
Verarbeite Dokumen

In [63]:
#pdf_bytes = pytesseract.image_to_pdf_or_hocr(img, lang='deu', config=custom_config, extension='pdf')

# 2. Daten als physische Datei speichern
#output_path = 'ausgabe_dokument.pdf'
#with open(output_path, 'wb') as f:
#    f.write(pdf_bytes)

#print(f"PDF erfolgreich unter '{output_path}' gespeichert.")